# 05 — IFPRI Standard CGE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/05_ifpri_standard_cge.ipynb)

CGE-Core contains an independently written Python/Pyomo implementation of the **IFPRI Standard CGE test economy**.

This notebook uses CGE-Core's **public synthetic IFPRI-format economy** so anyone can run it in Colab. The official IFPRI `test.dat` is intentionally not redistributed.

The conceptual step here is **closure**: different policy experiments may require different variables to absorb macroeconomic adjustment.

**v0.6 note:** IFPRI intentionally keeps its validated dedicated API in this release; the Hosoe `CGE` façade is not being forced onto this subsystem.


## 1. Setup

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# In Colab, set the environment variable CGE_CORE_REF to test a branch or tag.
# The public notebooks default to main. Outside Colab, a current CGE-Core git
# checkout is used directly, so branch development never silently resets to main.
CGE_CORE_REF = os.environ.get("CGE_CORE_REF", "main")
IN_COLAB = Path("/content").exists()

if not IN_COLAB and (Path.cwd() / ".git").is_dir() and (Path.cwd() / "cge_core").is_dir():
    REPO_DIR = Path.cwd()
    source_label = "current checkout"
else:
    WORKSPACE = Path("/content") if IN_COLAB else Path.home() / ".cache"
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = WORKSPACE / "CGE-core-colab"
    REPO_URL = "https://github.com/miraflor/CGE-core.git"

    if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a git checkout.")
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--no-checkout", REPO_URL, str(REPO_DIR)],
            check=True,
        )

    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", CGE_CORE_REF, "--depth", "1"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"],
        check=True,
    )
    source_label = CGE_CORE_REF

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import cge_core

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
print("✓ CGE-Core", cge_core.__version__)
print("✓ Source:", source_label, f"({commit})")
print("✓ Repository:", REPO_DIR)


import shutil

if not shutil.which("ipopt"):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "amplpy"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "amplpy.modules", "install", "coin"],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    module_path = subprocess.check_output(
        [sys.executable, "-m", "amplpy.modules", "path"],
        text=True,
    ).strip()
    os.environ["PATH"] = module_path + os.pathsep + os.environ.get("PATH", "")

assert shutil.which("ipopt"), "IPOPT was not found."
SOLVER = "ipopt"
print("✓ Solver:", SOLVER)


## 2. Load the redistributable IFPRI-format test economy

In [ ]:
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pyomo.environ import value

from cge_core.ifpri import (
    IfpriScenario,
    build_ifpri_base_solve_model,
    build_ifpri_scenario_model,
    calibrate_ifpri_benchmark,
    compare_ifpri_models,
    ifpri_degrees_of_freedom,
    solve_ifpri_base,
    solve_ifpri_scenario,
    validate_ifpri_calibration,
)

fixture_path = REPO_DIR / "tests" / "ifpri" / "synthetic.py"
spec = importlib.util.spec_from_file_location("cge_core_public_ifpri_synthetic", fixture_path)
synthetic = importlib.util.module_from_spec(spec)
spec.loader.exec_module(synthetic)

dataset = synthetic.build_synthetic_ifpri_dataset()

summary = pd.DataFrame(
    [
        ["Activities", ", ".join(dataset.sets.activities)],
        ["Commodities", ", ".join(dataset.sets.commodities)],
        ["Factors", ", ".join(dataset.sets.factors)],
        ["Households", ", ".join(dataset.sets.households)],
        ["Institutions", ", ".join(dataset.sets.institutions)],
    ],
    columns=["Set", "Members"],
)
display(summary)
print("SAM maximum absolute imbalance:", dataset.sam.max_abs_imbalance())

## 3. Algebraically calibrate the IFPRI benchmark

In [ ]:
calibration = calibrate_ifpri_benchmark(dataset)
validate_ifpri_calibration(dataset, calibration)

print("Benchmark foreign saving:", calibration.system.foreign_saving)
print("Benchmark government saving:", calibration.institutions.government_saving)
print("Import tariff on C:", calibration.taxes.import_["C"])

IFPRI calibration reconstructs normalized benchmark prices and quantities **before** the nonlinear equilibrium solve. This clean separation is useful for debugging larger CGE applications.

## 4. Solve the BASE closure

In [ ]:
base_model = build_ifpri_base_solve_model(dataset, calibration)
print("Degrees of freedom before solve:", ifpri_degrees_of_freedom(base_model))

base_report = solve_ifpri_base(base_model, SOLVER)

print("Termination:", base_report.termination_condition)
print("Degrees of freedom:", base_report.degrees_of_freedom)
print("Max equation residual:", base_report.max_abs_equation_residual)
print("Walras residual:", value(base_model.WALRAS))

## 5. Policy scenario: `TARCUT1`

`TARCUT1` cuts tariffs by 50% and uses a closure with **flexible government saving**.

The scenario therefore specifies both the shock and how the macro accounts are allowed to adjust.

In [ ]:
scenario = IfpriScenario.TARCUT1
scenario_model = build_ifpri_scenario_model(dataset, scenario)

print("Tariff on C:")
print("  BASE:", value(base_model.tm["C"]))
print("  TARCUT1 before solve:", value(scenario_model.tm["C"]))
print("Scenario degrees of freedom:", ifpri_degrees_of_freedom(scenario_model))

scenario_report = solve_ifpri_scenario(scenario_model, SOLVER)

print("Termination:", scenario_report.termination_condition)
print("Max equation residual:", scenario_report.max_abs_equation_residual)
print("Walras residual:", value(scenario_model.WALRAS))

## 6. Compare the two equilibria

In [ ]:
changes = compare_ifpri_models(
    base_model,
    scenario_model,
    scenario=scenario,
)

headline_components = ["EXR", "CPI", "QA", "QH", "QM", "QE"]
headline = changes[changes["component"].isin(headline_components)].copy()

display(
    headline[
        [
            "component", "index_1", "base_value", "scenario_value",
            "difference", "pct_change", "base_fixed", "scenario_fixed",
        ]
    ].style.format({
        "base_value": "{:.4f}",
        "scenario_value": "{:.4f}",
        "difference": "{:+.4f}",
        "pct_change": "{:+.2f}%",
    })
)

In [ ]:
trade = changes[changes["component"].isin(["QM", "QE", "QH"])].copy()

for component, title in [
    ("QM", "Imports"),
    ("QE", "Exports"),
    ("QH", "Domestic sales"),
]:
    part = trade[trade["component"] == component]
    if part.empty:
        continue
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(part["index_1"].astype(str), part["pct_change"].astype(float))
    ax.axhline(0, linewidth=0.8)
    ax.set_title(title + " — TARCUT1")
    ax.set_ylabel("% change from BASE")
    plt.show()

## The five IFPRI scenarios implemented in CGE-Core

| Scenario | Main experiment |
|---|---|
| `TARCUT1` | 50% tariff cut; government saving flexible |
| `TARCUT2` | 50% tariff cut; government saving fixed, direct tax adjusts |
| `FSAVINCR` | foreign saving +10% |
| `PWMINCR` | world import prices +10% |
| `DEVAL` | 10% devaluation under a fixed-exchange-rate closure |

Closure is therefore part of the policy experiment, not a technical afterthought.

## If you possess the official IFPRI test data

The repository intentionally does not redistribute `test.dat`.

With a separately obtained copy, point `IFPRI_SOURCE_DIR` at its folder and use:

```python
from cge_core.ifpri import load_ifpri_test_data
dataset = load_ifpri_test_data()
```

## Next

Notebook 06 asks whether CGE-Core can **reproduce a published historical CGE and its reported experiments**.

[Open Notebook 06 in Colab](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/06_camcge_replication.ipynb)